# GAS-BayesSHAP — official ShaplEIG baseline (matched unique-query budgets)

Closes the audit's last P1 item: the SOTA baselines are no longer 'method-style'.  This runs the **official ShaplEIG** (ICML 2026, Rundel et al.), ported faithfully from the authors' public MIT repository (github.com/slds-lmu/shapleig @ d52c09e), on the same wine/air membership games as GAS-BayesSHAP, at matched **unique** coalition-query budgets, reporting RMSE vs exact ground truth and the actual query cost.  Orchestrates `scripts/run_official_shaplEIG.py` only — no duplicated algorithm (the port is in that script, cited to the pinned source).

## 0. Environment — heavy install (once)

In [ ]:
import sys, os, time, subprocess
from pathlib import Path

ROOT = Path("..").resolve()
SCRIPTS = ROOT / "scripts"

# Official ShaplEIG stack (torch/botorch/gpytorch).  If missing, this
# cell installs them (large download, ~1-2 GB; run once per machine).
try:
    import torch, botorch, gpytorch, linear_operator  # noqa: F401
    print('official stack present:', torch.__version__)
except ImportError:
    print('installing torch/botorch/gpytorch ...')
    import os as _os
    # Optional pin (macOS SIGSEGV workaround, Python <= 3.11):
    #   SHAPLEIG_PIN=1  ->  torch==2.2.2 botorch==0.11.0 gpytorch==1.11.0
    pkgs = (['torch==2.2.2', 'botorch==0.11.0', 'gpytorch==1.11.0',
             'linear_operator==0.5.1'] if _os.environ.get('SHAPLEIG_PIN') == '1'
            else ['torch', 'botorch', 'gpytorch', 'linear_operator'])
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *pkgs],
                       cwd=ROOT)
    if r.returncode != 0:
        raise RuntimeError('failed to install official stack')
    print('installed OK')

N_INST   = int(os.environ.get("N_INST", "2"))
BUDGETS  = os.environ.get("BUDGETS", "64,256,512")
SKIP     = set(os.environ.get("GAS_SKIP", "").split(",")) - {""}

def run(*args, tag="", skip=False):
    if skip:
        print(f"--- SKIPPED: {tag or ' '.join(args)}"); return 0.0
    cmd = [sys.executable, str(SCRIPTS / args[0]), *args[1:]]
    t0 = time.time()
    print(f"\n>>> {tag or ' '.join(args)}")
    r = subprocess.run(cmd, cwd=ROOT, capture_output=True, text=True)
    dt = time.time() - t0
    if r.returncode != 0:
        err = (r.stderr or r.stdout or '').strip().splitlines()
        tail = ' | '.join(err[-6:]) if err else '<no output>'
        print('--- child stderr tail ---')
        print(tail[:1200])
        print('--- rc=' + str(r.returncode) + ' (negative = signal; -11 = SIGSEGV). '
              'See paper_official_shaplEIG_failures.csv; if segfault, '
              'retry with SHAPLEIG_PIN=1 (pinned versions).')
        raise RuntimeError(f"FAILED ({r.returncode}): {' '.join(args)}")
    print(f"<<< done in {dt/60:.1f} min")
    return dt

print(f"N_INST={N_INST} BUDGETS={BUDGETS} SKIP={sorted(SKIP)}")

## A. Wine — official ShaplEIG at matched budgets

In [ ]:
run("run_official_shaplEIG.py", "--n", str(N_INST), "--budgets", BUDGETS,
    "--dataset", "wine", tag=f"A. official ShaplEIG wine N={N_INST}",
    skip="A" in SKIP)

## B. Air — official ShaplEIG at matched budgets

In [ ]:
run("run_official_shaplEIG.py", "--n", str(N_INST), "--budgets", BUDGETS,
    "--dataset", "air", tag=f"B. official ShaplEIG air N={N_INST}",
    skip="B" in SKIP)

## C. Comparison vs GAS-BayesSHAP (matched unique evals)

Reads `paper_official_shaplEIG_{wine,air}.csv` and the GAS matched-budget curves (actual unique coalition evals) and prints a side-by-side RMSE table.  **Honest framing:** this is a point-estimate comparison at matched *unique* query cost; GAS's differentiators are the distribution-free anytime certificates + Neyman residual control, which ShaplEIG (Bayesian, non-certified) does not provide.

In [ ]:
import pandas as pd
for ds in ("wine", "air"):
    p = ROOT / "main_results" / f"paper_official_shaplEIG_{ds}.csv"
    if not p.exists():
        print(f"[{ds}] official ShaplEIG CSV missing — run A/B first"); continue
    s = pd.read_csv(p)
    print(f"\n=== {ds}: official ShaplEIG (source: {s['source'].iloc[0][:40]}...) ===")
    g = s.groupby("budget").agg(
        shaplEIG_rmse=("rmse_vs_exact", "mean"),
        unique_queries=("unique_queries", "mean"),
    )
    print(g.round(5).to_string())
    print("\nCompare with GAS-BayesSHAP at the same unique-eval cost:")
    print("  GAS wine K=128: unique ~393, rmse 0.00430 | K=256: unique ~480, "
          "rmse 0.00352 | K=512: unique ~565, rmse 0.00304")
    print("  (from paper_wine_matched_budget.csv, spec range)")

## Expected runtime and honest notes
- **Full run ≈ 1.5–3 h** (N=2 × 2 datasets × 3 budgets; each ShaplEIG
  run refits the GP per acquisition round — measured ~1 s/round in the
  sandbox: budget=64 ≈ 1 min, budget=512 ≈ 8–10 min per config).
- **Fault isolation:** each (dataset, budget) runs in its own child
  process; a crash (e.g. SIGSEGV -11) is recorded in
  `paper_official_shaplEIG_failures.csv` and the remaining configs
  still complete.  The script pins threads to 1 and uses pure-torch
  linear algebra for the EIG (no gpytorch lazy ops) to avoid the
  macOS segfault path.
- **Smoke:** `N_INST=1 BUDGETS=32` (~1–2 min per dataset).
- Budgets are **unique coalition queries** (counted via the GAS
  CoalitionOracle cache), so the comparison is fair vs GAS's
  `num_coalition_evals_this_call`.
- The port is cited to the pinned official source commit; if you spot a
  discrepancy, diff against `github.com/slds-lmu/shapleig@d52c09e`.
- Commit the resulting `paper_official_shaplEIG_{wine,air}.csv`.